# Python 推导式详解（Comprehensions）

> **一句话本质**：推导式是"把循环和条件压缩成一个表达式"的工具——简单转换用它，复杂逻辑用循环。

## 学习目标

学完本 Notebook，你将能够：

1. 熟练写出 **列表 / 字典 / 集合 / 生成器** 四种推导式，并理解各自的求值时机；
2. 区分 `if` 在 `for` **前**（条件表达式 / 映射）与 `for` **后**（过滤）的两种语义；
3. 理解推导式的 **独立作用域**，以及生成器表达式 **惰性求值** 带来的内存优势；
4. 识别并规避 7 类常见陷阱（闭包延迟绑定、生成器耗尽、键覆盖等）；
5. 在 **LLMOps API 开发**（RAG 文档清洗、元数据索引、日志惰性统计）中恰当地选择推导式或普通循环。

## 学习路径

前置知识 → 四种推导式 → 作用域与性能 → 反例 → 工程模式 → 综合实战 → 陷阱汇总 → 总结自测


## 0. 前置知识

**本章解决什么问题**：推导式是"语法糖"，理解它之前必须先确认四个语言基础——可迭代对象、for 循环、条件表达式（三元表达式）、`zip` / `dict.items()` / `set` 的语义。下面用一个可运行单元逐一验证。

| 前置概念 | 一句话说明 | 在推导式中的角色 |
| --- | --- | --- |
| 可迭代对象（Iterable） | 实现了 `__iter__` 的对象，如 list / str / range / dict | `for 变量 in 可迭代对象` 的数据来源 |
| for 循环 | 逐个取出元素执行循环体 | 推导式的"骨架" |
| 条件表达式 | `A if 条件 else B`，是一个**表达式**，有返回值 | 放在 `for` 前面做"映射" |
| `zip` / `dict.items()` / `set` | 并行遍历 / 键值对遍历 / 去重集合 | 字典与集合推导式的常用数据源 |


In [1]:
# 前置知识验证：四个基础概念逐一跑通

# 0.1 可迭代对象：list / str / range / dict 都可以被 for 遍历
for ch in "abc":
    pass  # 字符串本身就是可迭代对象
print("0.1 可迭代对象验证:", list("abc"), list(range(3)), list({"k1": 1}))  # dict 默认迭代键

# 0.2 条件表达式（三元表达式）：有返回值，可以赋给变量
n = 7
label = "奇数" if n % 2 == 1 else "偶数"  # 条件为真取前者，否则取后者
print("0.2 条件表达式验证:", n, "->", label)

# 0.3 zip：把两个序列按位置配对，短的先耗尽就停止
keys = ["a", "b", "c"]
values = [1, 2, 3]
print("0.3 zip 验证:", list(zip(keys, values)))

# 0.4 dict.items() 拿到 (键, 值) 元组；set 自动去重
d = {"x": 10, "y": 20}
print("0.4 items/set 验证:", list(d.items()), set([1, 2, 2, 3]))


0.1 可迭代对象验证: ['a', 'b', 'c'] [0, 1, 2] ['k1']
0.2 条件表达式验证: 7 -> 奇数
0.3 zip 验证: [('a', 1), ('b', 2), ('c', 3)]
0.4 items/set 验证: [('x', 10), ('y', 20)] {1, 2, 3}


## 1. 为什么需要推导式

**本章解决什么问题**：同一件事，普通 `for + append` 要写 3 行，推导式 1 行——而且语义更集中、通常更快。本章用等价性验证代码证明：**两种写法结果完全一致**。

| 维度 | 普通 for + append | 推导式 |
| --- | --- | --- |
| 代码量 | 3 行 | 1 行 |
| 语义 | 过程式（"怎么做"） | 声明式（"要什么"） |
| 速度 | 慢（每次 `append` 方法调用） | 通常快 20%~50%（第 7 章验证） |
| 适用 | 复杂逻辑、副作用 | 映射 / 过滤 / 转换 |


In [2]:
# 知识点1：推导式与普通 for 循环的等价性验证

# 1.1 普通写法：先建空列表，循环里逐个 append
result_loop = []
for x in range(10):
    result_loop.append(x * x)  # 每次循环调用一次 append 方法

# 1.2 推导式写法：一句话声明"我要每个 x 的平方组成的列表"
result_comp = [x * x for x in range(10)]

# 1.3 用 assert 证明两者完全等价（不抛异常即相等）
assert result_loop == result_comp, "两种写法结果不一致！"
print("普通 for 写法:", result_loop)
print("推导式写法  :", result_comp)
print("等价性验证  : 通过（assert 未触发）")


普通 for 写法: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
推导式写法  : [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
等价性验证  : 通过（assert 未触发）


## 2. 列表推导式

**本章解决什么问题**：掌握列表推导式的四种形态——基本映射、条件过滤、条件表达式、嵌套循环。

### 通用语法骨架

```python
[表达式 for 变量 in 可迭代对象 if 条件]
```

### 关键辨析：`if` 的位置决定语义

| 写法 | `if` 位置 | 语义 | 结果长度 |
| --- | --- | --- | --- |
| `[x for x in nums if x > 0]` | `for` **后** | **过滤**：不满足的元素直接丢弃 | ≤ 原长度 |
| `[f(x) if 条件 else g(x) for x in nums]` | `for` **前** | **映射**：每个元素二选一取值 | = 原长度 |

> 记忆法：`if` 在 `for` 后面没有 `else`，在 `for` 前面必须有 `else`。


In [3]:
# 知识点2.1：列表推导式基本语法 —— [表达式 for 变量 in 可迭代对象]

# 示例1：生成 0~4 的平方
squares = [x * x for x in range(5)]
print("平方列表:", squares)  # [0, 1, 4, 9, 16]

# 示例2（LLMOps 场景）：把一批用户输入的 prompt 统一转成大写做规范化
names = ["alice", "bob", "carol"]
upper_names = [name.upper() for name in names]
print("大写转换:", upper_names)  # ['ALICE', 'BOB', 'CAROL']


平方列表: [0, 1, 4, 9, 16]
大写转换: ['ALICE', 'BOB', 'CAROL']


In [4]:
# 知识点2.2：带条件过滤 —— if 放在 for 后面，元素个数会减少

# 示例1：筛出 0~9 中的偶数
evens = [x for x in range(10) if x % 2 == 0]
print("偶数过滤:", evens)  # [0, 2, 4, 6, 8]

# 示例2（LLMOps 场景）：清洗文档列表，丢弃空字符串（空串是"假值"，if w 直接过滤）
words = ["hello", "", "world", "", "python"]
non_empty = [w for w in words if w]
print("非空过滤:", non_empty)  # ['hello', 'world', 'python']


偶数过滤: [0, 2, 4, 6, 8]
非空过滤: ['hello', 'world', 'python']


In [5]:
# 知识点2.3：条件表达式 —— if/else 放在 for 前面，是"二选一映射"，元素个数不变

# 每个数字映射为"偶数"/"奇数"标签
labels = ["偶数" if x % 2 == 0 else "奇数" for x in range(5)]
print("条件映射:", labels)  # ['偶数', '奇数', '偶数', '奇数', '偶数']

# 关键辨析验证：过滤（for 后） vs 映射（for 前），输出长度不同
filtered = [x for x in range(10) if x % 2 == 0]              # 过滤：长度 5
mapped = ["偶" if x % 2 == 0 else "奇" for x in range(3)]     # 映射：长度 3
print("过滤结果（长度变短）:", filtered, "长度 =", len(filtered))
print("映射结果（长度不变）:", mapped, "长度 =", len(mapped))


条件映射: ['偶数', '奇数', '偶数', '奇数', '偶数']
过滤结果（长度变短）: [0, 2, 4, 6, 8] 长度 = 5
映射结果（长度不变）: ['偶', '奇', '偶'] 长度 = 3


In [6]:
# 知识点2.4：嵌套循环 —— 多个 for 从左到右等价于 for 的嵌套层次

# 一行生成笛卡尔积
pairs = [(x, y) for x in range(2) for y in range(2)]
print("嵌套循环:", pairs)  # [(0, 0), (0, 1), (1, 0), (1, 1)]

# 等价写法：外层 for 对应左边，内层 for 对应右边
pairs_loop = []
for x in range(2):       # 外层：与推导式左边的 for 对应
    for y in range(2):   # 内层：与推导式右边的 for 对应
        pairs_loop.append((x, y))

assert pairs == pairs_loop, "嵌套推导式与双层 for 不等价！"
print("等价性验证: 通过（左侧 for 是外层）")


嵌套循环: [(0, 0), (0, 1), (1, 0), (1, 1)]
等价性验证: 通过（左侧 for 是外层）


In [7]:
# 知识点2.5：嵌套推导式 —— 扁平化与矩阵转置

# 示例1：二维列表扁平化（RAG 分块后常见：多个文档的 chunk 列表合成一个列表）
matrix = [[1, 2], [3, 4], [5, 6]]
flat = [n for row in matrix for n in row]  # 先取行，再取行内元素
print("扁平化:", flat)  # [1, 2, 3, 4, 5, 6]

# 示例2：矩阵转置 —— 外层按列号 i，内层取每行的第 i 个元素
matrix2 = [[1, 2, 3], [4, 5, 6]]
transposed = [[row[i] for row in matrix2] for i in range(3)]
print("矩阵转置:", transposed)  # [[1, 4], [2, 5], [3, 6]]

# 注意：超过两层的嵌套推导式可读性会急剧下降，工程上建议拆成普通循环


扁平化: [1, 2, 3, 4, 5, 6]
矩阵转置: [[1, 4], [2, 5], [3, 6]]


In [8]:
# 知识点2.6：多个条件 —— 多个 if 是"与"关系

# 筛出 0~19 中同时被 2 和 3 整除的数
nums = [x for x in range(20) if x % 2 == 0 if x % 3 == 0]
print("多 if 过滤:", nums)  # [0, 6, 12, 18]

# 等价写法：用 and 合并成一个条件
nums_and = [x for x in range(20) if x % 2 == 0 and x % 3 == 0]
assert nums == nums_and, "多个 if 与 and 合并不等价！"
print("等价性验证: 多个 if == and 连接，通过")


多 if 过滤: [0, 6, 12, 18]
等价性验证: 多个 if == and 连接，通过


## 3. 字典推导式

**本章解决什么问题**：快速构建"键 → 值"映射，是 LLMOps 中构建**索引、配置映射、字段重命名**的主力工具。

### 语法骨架

```python
{键表达式: 值表达式 for 变量 in 可迭代对象 if 条件}
```

### 核心要点

- 常与 `zip(键序列, 值序列)` 或 `dict.items()` 配合；
- **反转字典时键必须唯一**——值有重复时，后出现的会覆盖先出现的（下面用代码验证）；
- 复杂分组逻辑（如 `setdefault` 累加）不适合强行用推导式表达。


In [9]:
# 知识点3.1~3.2：字典推导式基本语法 与 从两个序列构建

# 3.1 基本：数字 -> 它的平方
squares = {x: x * x for x in range(5)}
print("基本语法:", squares)  # {0: 0, 1: 1, 2: 4, 3: 9, 4: 16}

# 3.2 用 zip 把两个序列拼成字典（LLMOps 场景：字段名 -> 默认值 的配置映射）
keys = ["temperature", "max_tokens", "top_p"]
values = [0.7, 2048, 1.0]
llm_config = {k: v for k, v in zip(keys, values)}
print("zip 构建:", llm_config)


基本语法: {0: 0, 1: 1, 2: 4, 3: 9, 4: 16}
zip 构建: {'temperature': 0.7, 'max_tokens': 2048, 'top_p': 1.0}


In [10]:
# 知识点3.3：反转字典 —— 值变键、键变值，注意"重复值覆盖"陷阱

# 情况1：值唯一，反转是安全的
d = {"a": 1, "b": 2, "c": 3}
inv = {v: k for k, v in d.items()}
print("值唯一时反转:", inv)  # {1: 'a', 2: 'b', 3: 'c'}

# 情况2：值有重复，后出现的键会覆盖先出现的（"a" 被 "b" 顶掉）
d2 = {"a": 1, "b": 1, "c": 2}
inv2 = {v: k for k, v in d2.items()}
print("值重复时反转:", inv2)  # {1: 'b', 2: 'c'} —— "a" 丢失！
print("覆盖验证:", "键 'a' 已丢失" if "a" not in inv2.values() else "未覆盖")


值唯一时反转: {1: 'a', 2: 'b', 3: 'c'}
值重复时反转: {1: 'b', 2: 'c'}
覆盖验证: 键 'a' 已丢失


In [11]:
# 知识点3.4~3.6：过滤、交换并过滤、分组

# 3.4 过滤：只保留偶数值的条目
d = {"a": 1, "b": 2, "c": 3}
even = {k: v for k, v in d.items() if v % 2 == 0}
print("值过滤:", even)  # {'b': 2}

# 3.5 交换键值并同时过滤
result = {v: k for k, v in d.items() if v > 1}
print("交换+过滤:", result)  # {2: 'b', 3: 'c'}

# 3.6 分组（LLMOps 场景：按首字母给文档分桶）
words = ["apple", "banana", "avocado", "cherry"]
# 外层：对每个首字母建键；内层：列表推导式收集该字母开头的所有词
groups = {letter: [w for w in words if w.startswith(letter)]
          for letter in {w[0] for w in words}}  # 集合推导式先去重出所有首字母
print("分组结果:", groups)


值过滤: {'b': 2}
交换+过滤: {2: 'b', 3: 'c'}
分组结果: {'a': ['apple', 'avocado'], 'c': ['cherry'], 'b': ['banana']}


## 4. 集合推导式

**本章解决什么问题**：在"收集结果的同时自动去重"。

### 语法骨架

```python
{表达式 for 变量 in 可迭代对象 if 条件}
```

### 与列表推导式的差异

| 维度 | 列表推导式 `[...]` | 集合推导式 `{...}` |
| --- | --- | --- |
| 重复元素 | 保留 | **自动去重** |
| 元素顺序 | 保持插入顺序 | 不保证业务意义上的顺序 |
| 典型用途 | 结果要保序、可重复 | 只关心"有哪些种" |


In [12]:
# 知识点4.1~4.3：集合推导式 —— 去重是核心价值

# 4.1 基本语法：负数和正数的平方相同，集合自动去重
squares = {x * x for x in range(-3, 4)}
print("平方集合:", sorted(squares))  # [0, 1, 4, 9] —— 排序后输出保证可复现

# 4.1 补充（LLMOps 场景）：统计语料中出现过哪几种文本长度
words = ["hello", "world", "hello", "python"]
unique_lengths = {len(w) for w in words}
print("长度种类:", sorted(unique_lengths))  # [5, 6]

# 4.2 带过滤
nums = {x for x in range(20) if x % 3 == 0}
print("3 的倍数:", sorted(nums))  # [0, 3, 6, 9, 12, 15, 18]

# 4.3 保持顺序去重：集合推导式不保序，要用 dict.fromkeys（键唯一且保序）
tags = ["rag", "agent", "rag", "memory", "agent"]
unique_tags = list(dict.fromkeys(tags))
print("保序去重:", unique_tags)  # ['rag', 'agent', 'memory']


平方集合: [0, 1, 4, 9]
长度种类: [5, 6]
3 的倍数: [0, 3, 6, 9, 12, 15, 18]
保序去重: ['rag', 'agent', 'memory']


## 5. 生成器表达式

**本章解决什么问题**：当数据量大到"全部装进内存不划算"时，用生成器表达式**按需计算**——这是它与列表推导式最本质的区别。

### 语法骨架

```python
(表达式 for 变量 in 可迭代对象 if 条件)
```

### 与列表推导式对比

| 维度 | 列表推导式 `[...]` | 生成器表达式 `(...)` |
| --- | --- | --- |
| 求值时机 | **立即**求值，全部入内存 | **惰性**求值，用多少算多少 |
| 内存占用 | 与元素个数成正比 | 固定约 200 字节（下面验证） |
| 可重复遍历 | 可以 | **只能消费一次** |
| 优势 | 速度快、可索引 | 省内存、可对接管道 |


In [13]:
# 知识点5.1：生成器表达式是惰性求值 —— 用 next 逐个"要"值

# 注意外层是圆括号
gen = (x * x for x in range(5))
print("类型:", type(gen).__name__)  # generator —— 只是"配方"，还没有任何计算发生

# 每次 next 才计算一个元素
print("第1次 next:", next(gen))  # 0
print("第2次 next:", next(gen))  # 1

# 剩余的用 list 一次性收走
print("剩余元素:", list(gen))  # [4, 9, 16]


类型: generator
第1次 next: 0
第2次 next: 1
剩余元素: [4, 9, 16]


In [14]:
# 知识点5.2：内存占用对比 —— 用 sys.getsizeof 量化验证

import sys

# 列表推导式：100 万个结果立刻全部算好存进内存
lst = [x * x for x in range(1_000_000)]
# 生成器表达式：只存"怎么算"的配方，不存结果
gen = (x * x for x in range(1_000_000))

size_lst = sys.getsizeof(lst)
size_gen = sys.getsizeof(gen)
print(f"列表占用: {size_lst:,} 字节 ≈ {size_lst / 1024 / 1024:.1f} MB")
print(f"生成器占用: {size_gen:,} 字节")
print(f"内存倍数: 列表约是生成器的 {size_lst // size_gen:,} 倍")


列表占用: 8,448,728 字节 ≈ 8.1 MB
生成器占用: 200 字节
内存倍数: 列表约是生成器的 42,243 倍


In [15]:
# 知识点5.3：函数调用中省略括号 —— 生成器是唯一参数时可省一层 ()

# 推荐写法：sum 直接吃生成器，不建中间列表
total = sum(x * x for x in range(10))
print("生成器求和:", total)  # 285

# 对比写法：sum([...]) 会先建一个完整列表，白白多用一份内存
total2 = sum([x * x for x in range(10)])
assert total == total2, "两种求和结果不一致！"
print("结果等价:", total2, "（但生成器写法更省内存）")

# 同样适用于 max / min / any / all 等聚合函数
max_len = max(len(w) for w in ["hello", "hi", "python"])
print("最长字符串长度:", max_len)  # 6


生成器求和: 285
结果等价: 285 （但生成器写法更省内存）
最长字符串长度: 6


In [16]:
# 知识点5.4：处理大文件 —— 逐行惰性读取，内存占用与文件大小无关

import os
import tempfile

# 构造一个确定性的临时文件（模拟 LLMOps 的应用日志）
log_path = os.path.join(tempfile.gettempdir(), "llmops_app.log")
with open(log_path, "w", encoding="utf-8") as f:
    for i in range(1, 6):  # 写入 5 行确定性内容
        f.write(f"2026-09-30 INFO request-{i} tokens={i * 100}\n")

# 关键：文件对象本身是可迭代的，sum(len(line) for line in f) 逐行统计
with open(log_path, encoding="utf-8") as f:
    total_chars = sum(len(line) for line in f)  # 不会把整个文件读进内存

print("日志总字符数:", total_chars)
print("逐行验证:", sum(len(f"2026-09-30 INFO request-{i} tokens={i * 100}\n") for i in range(1, 6)) == total_chars)

os.remove(log_path)  # 清理临时文件


日志总字符数: 185
逐行验证: True


## 6. 推导式的作用域

**本章解决什么问题**：Python 3 中推导式有**独立作用域**，循环变量不会"泄漏"到外部；但普通 `for` 循环**会泄漏**。这个差异在调试时经常造成困惑，下面用输出直接证明。

### 要点

1. 列表 / 字典 / 集合 / 生成器推导式：循环变量是局部的，外部同名变量不受影响；
2. 普通 `for` 循环：循环变量会**覆盖**外部同名变量；
3. **例外**：在类体（class body）中，推导式无法访问类属性——它的作用域链跳过类作用域直接找全局（第 9 章反例会触发真实 `NameError`）。


In [17]:
# 知识点6.1：作用域验证 —— 推导式不泄漏，普通 for 会泄漏

# 实验1：推导式不污染外部变量
x = 10
lst = [x for x in range(3)]  # 这里的 x 是推导式内部的局部变量
print("推导式执行后外部 x:", x)  # 10 —— 没有被修改！

# 实验2：普通 for 循环会覆盖外部变量
x = 10
for x in range(3):
    pass  # 循环结束后，x 停留在最后一个值
print("for 循环执行后外部 x:", x)  # 2 —— 被覆盖了！

# 实验3：生成器表达式同样有自己的作用域
x = 10
gen = (x for x in range(3))
print("生成器表达式执行后外部 x:", x)  # 10


推导式执行后外部 x: 10
for 循环执行后外部 x: 2
生成器表达式执行后外部 x: 10


In [18]:
# 知识点6.2（反例）：类体中的推导式读不到类属性 —— 先触发报错，再给正确写法

# 错误写法：推导式的作用域链会"跳过"类作用域直接找全局，因此触发 NameError
try:
    class A:
        x = 10
        lst = [x for _ in range(3)]  # 这里的 x：不在全局，且推导式跳过类作用域，找不到
except NameError as e:
    print("错误写法触发 NameError:", e)

# 正确写法：类定义完成后，通过类名显式引用再赋值（工程中最清晰的做法）
class D:
    x = 10

D.lst = [D.x for _ in range(3)]  # 显式写 D.x，绕开类作用域查找问题
print("正确写法输出:", D.lst)  # [10, 10, 10]


正确写法输出: [10, 10, 10]


## 7. 性能对比

**本章解决什么问题**：推导式"通常更快"不是玄学——底层用 `LIST_APPEND` 等专用字节码，省掉了方法查找开销。用 `timeit` 实测验证。

### 预期结论

- 列表推导式比 `append` 循环快约 **20%~50%**；
- 生成器表达式的优势在**内存**而不在速度——逐个消费时它不一定比列表快。


In [19]:
# 知识点7：timeit 实测 —— 推导式 vs append 循环

import timeit

def loop():
    result = []
    for x in range(1000):
        result.append(x * x)  # 每次都要做 append 属性查找 + 方法调用
    return result

def comp():
    return [x * x for x in range(1000)]  # 编译成专用字节码，省去方法调用开销

t_loop = timeit.timeit(loop, number=10000)
t_comp = timeit.timeit(comp, number=10000)

print(f"append 循环: {t_loop:.3f} 秒")
print(f"列表推导式: {t_comp:.3f} 秒")
print(f"推导式提速: {(t_loop - t_comp) / t_loop * 100:.1f}%")
print("结论验证:", "推导式更快" if t_comp < t_loop else "本次运行未体现差异（可多跑几次）")


append 循环: 0.205 秒
列表推导式: 0.180 秒
推导式提速: 12.1%
结论验证: 推导式更快


## 8. 推导式与 map / filter 对比

**本章解决什么问题**：同一件事有两种风格，什么时候用哪个？

| 场景 | 推荐 | 理由 |
| --- | --- | --- |
| 简单映射 + 过滤 | 推导式 | 一个表达式写完，不用嵌套两层 lambda |
| 已有现成函数（如 `str`、`int`） | `map` / `filter` | `map(str, nums)` 比 `[str(x) for x in nums]` 还短 |
| 需要惰性、大数据 | 生成器表达式 / `map` | 两者都是惰性的 |
| 复杂多步逻辑 | 普通 `for` 循环 | 可读性优先 |


In [20]:
# 知识点8：推导式与 map/filter 的等价性验证

nums = range(10)

# 写法1：列表推导式 —— 一行同时完成过滤和映射
result1 = [x * 2 for x in nums if x % 2 == 0]

# 写法2：map/filter —— filter 先过滤，map 再映射，最外层 list 收结果
result2 = list(map(lambda x: x * 2, filter(lambda x: x % 2 == 0, nums)))

assert result1 == result2, "两种风格结果不一致！"
print("推导式结果 :", result1)
print("map/filter :", result2)
print("等价性验证 : 通过")

# 已有现成函数时，map 更简洁：不需要写 lambda
codes = [65, 66, 67]
print("map 配现成函数:", list(map(chr, codes)))  # ['A', 'B', 'C']


推导式结果 : [0, 4, 8, 12, 16]
map/filter : [0, 4, 8, 12, 16]
等价性验证 : 通过
map 配现成函数: ['A', 'B', 'C']


## 9. 反例单元：先犯错，再纠正

**本章解决什么问题**：每个易错点先给出**错误写法**并实际触发报错（或错误输出），再给出**正确写法**对比——亲眼看到差异比背规则有效得多。

> 注：报错用 `try/except` 捕获并打印，保证 Notebook 整体 0 未处理异常。


In [21]:
# 反例1：生成器只能消费一次 —— 第二次遍历拿到空结果

gen = (x for x in range(3))
print("错误写法第1次 list:", list(gen))  # [0, 1, 2] —— 正常
print("错误写法第2次 list:", list(gen))  # [] —— 已经耗尽，静默拿到空列表！

# 正确写法：需要多次使用时，一开始就建成列表（数据量小）或重新创建生成器
data = [x for x in range(3)]
print("正确写法第1次:", list(data))  # [0, 1, 2]
print("正确写法第2次:", list(data))  # [0, 1, 2] —— 列表可反复遍历


错误写法第1次 list: [0, 1, 2]
错误写法第2次 list: []
正确写法第1次: [0, 1, 2]
正确写法第2次: [0, 1, 2]


In [22]:
# 反例2：闭包延迟绑定 —— lambda 记住的是"变量"，不是"当时的值"

# 错误写法：三个 lambda 共享同一个循环变量 x，调用时才取值（此时 x 已是 2）
funcs_bad = [lambda: x for x in range(3)]
print("错误写法输出:", [f() for f in funcs_bad])  # [2, 2, 2] —— 全是最后一个值！

# 正确写法：用默认参数 x=x 把"当时的值"绑定进每个 lambda
funcs_good = [lambda x=x: x for x in range(3)]
print("正确写法输出:", [f() for f in funcs_good])  # [0, 1, 2]


错误写法输出: [2, 2, 2]
正确写法输出: [0, 1, 2]


In [23]:
# 反例3：在推导式里做副作用 —— 能运行，但生成一个无意义的 [None, None, None]

# 错误写法：print 返回 None，推导式收集到的是一堆 None
side_effects = [print("处理:", x) for x in range(3)]
print("错误写法的返回值:", side_effects)  # [None, None, None]

# 正确写法：副作用操作就该用普通 for 循环，语义诚实
for x in range(3):
    print("处理:", x)


处理: 0
处理: 1
处理: 2
错误写法的返回值: [None, None, None]
处理: 0
处理: 1
处理: 2


In [24]:
# 反例4：在推导式中修改外部状态（海象运算符 :=）—— 合法但强烈不推荐

total = 0
# 错误写法：推导式的本职是"生成值"，这里却偷偷累加外部变量，列表本身被丢弃
[total := total + x for x in range(5)]
print("错误写法累加结果:", total)  # 10 —— 结果对，但意图被藏起来了

# 正确写法：求和是聚合操作，用 sum 一目了然
total = sum(range(5))
print("正确写法累加结果:", total)  # 10


错误写法累加结果: 10
正确写法累加结果: 10


## 10. 实用示例集：LLMOps 高频模式

**本章解决什么问题**：把推导式落到 LLMOps API 开发的 5 个真实场景。每个模式给出适用场景与注意点。

| # | 模式 | 适用场景 | 注意点 |
| --- | --- | --- | --- |
| 1 | 清洗 + 过滤一体 | 用户输入 / 文档语料清洗 | `if s.strip()` 调两次，量大时先赋值再过滤 |
| 2 | 字段提取 + 条件筛选 | 从 dict 列表中取子集 | 等价于 SQL 的 `SELECT ... WHERE` |
| 3 | 构建 ID 索引 | 按主键快速查找记录 | 键重复时后覆盖前 |
| 4 | 词频统计 | 语料分析、关键词提取 | 小规模用推导式，生产用 `Counter` |
| 5 | 惰性管道 | 大日志 / 大文件统计 | 全程不落地中间列表 |


In [25]:
# 模式1：数据清洗 —— strip 去空白 + 过滤空串，一步到位

# LLMOps 场景：清洗一批用户提交的 query，去掉首尾空白并丢弃纯空白输入
raw_queries = ["  什么是RAG ", "", " 帮我写个Agent", "   ", "LangChain memory 怎么用  "]
clean_queries = [s.strip() for s in raw_queries if s.strip()]  # 先过滤非空，再收集 strip 结果
print("清洗前:", raw_queries)
print("清洗后:", clean_queries)


清洗前: ['  什么是RAG ', '', ' 帮我写个Agent', '   ', 'LangChain memory 怎么用  ']
清洗后: ['什么是RAG', '帮我写个Agent', 'LangChain memory 怎么用']


In [26]:
# 模式2：字段提取 + 条件筛选 —— 相当于 SQL 的 SELECT/WHERE

# LLMOps 场景：从文档元数据列表中提取标题，并筛出高相关度文档
docs = [
    {"title": "RAG 检索入门", "score": 0.92},
    {"title": "Agent 设计模式", "score": 0.61},
    {"title": "Memory 组件解析", "score": 0.85},
]

titles = [d["title"] for d in docs]                       # SELECT title
high_recall = [d for d in docs if d["score"] >= 0.8]      # WHERE score >= 0.8
print("全部标题:", titles)
print("高相关度文档:", [d["title"] for d in high_recall])


全部标题: ['RAG 检索入门', 'Agent 设计模式', 'Memory 组件解析']
高相关度文档: ['RAG 检索入门', 'Memory 组件解析']


In [27]:
# 模式3：构建 ID 索引 —— O(n) 建一次，之后 O(1) 查找

# LLMOps 场景：API 返回的文档列表按 doc_id 建索引，避免每次线性查找
documents = [
    {"doc_id": "doc-001", "title": "RAG 检索入门"},
    {"doc_id": "doc-002", "title": "Agent 设计模式"},
]
by_id = {d["doc_id"]: d for d in documents}  # 键: doc_id，值: 整条记录
print("索引键:", list(by_id.keys()))
print("O(1) 查找:", by_id["doc-001"]["title"])


索引键: ['doc-001', 'doc-002']
O(1) 查找: RAG 检索入门


In [28]:
# 模式4：词频统计 —— 推导式版 vs Counter 版

text = "the quick brown fox jumps over the lazy dog"

# 写法1：推导式版 —— 直观但 count 是 O(n)，总复杂度 O(n²)
freq1 = {word: text.split().count(word) for word in set(text.split())}

# 写法2：Counter 版 —— 一趟遍历 O(n)，生产环境推荐
from collections import Counter
freq2 = Counter(text.split())

assert dict(freq2) == freq1, "两种词频统计结果不一致！"
print("推导式版 the 出现次数:", freq1["the"])   # 2
print("Counter 版最常见的词:", freq2.most_common(1))  # [('the', 2)]
print("等价性验证: 通过")


推导式版 the 出现次数: 2
Counter 版最常见的词: [('the', 2)]
等价性验证: 通过


In [29]:
# 模式5：惰性管道 —— 大日志中统计 token 消耗，全程不落地中间列表

import os
import tempfile

# 构造确定性的模拟日志（每行一条 token 消耗记录，含 2 条无效行）
log_path = os.path.join(tempfile.gettempdir(), "llmops_tokens.log")
lines = ["tokens=120\n", "tokens=350\n", "invalid-line\n", "tokens=80\n", "\n"]
with open(log_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

# 管道思路：逐行读 -> 过滤无效行 -> 解析数字 -> 聚合，每步都是惰性的
with open(log_path, encoding="utf-8") as f:
    token_counts = (int(line.split("=")[1])          # 步骤3：解析出 token 数
                    for line in f                     # 步骤1：逐行惰性读取
                    if line.startswith("tokens="))    # 步骤2：过滤无效行
    total_tokens = sum(token_counts)                  # 步骤4：聚合时才真正驱动计算

print("Token 总消耗:", total_tokens)  # 120 + 350 + 80 = 550
os.remove(log_path)  # 清理临时文件


Token 总消耗: 550


## 11. 内置 / 生态对照

**本章解决什么问题**：推导式不是孤立的——标准库和第三方生态里有大量"同赛道"能力，选型时对照下表。

| 能力 | 推导式写法 | 内置 / 标准库替代 | 第三方生态 | 选型建议 |
| --- | --- | --- | --- | --- |
| 映射 | `[f(x) for x in it]` | `map(f, it)` | NumPy 向量化 `arr * 2` | 简单逻辑用推导式；数值计算用 NumPy |
| 过滤 | `[x for x in it if p(x)]` | `filter(p, it)` / `itertools.compress` | pandas 布尔索引 | 推导式可读性最好 |
| 保序去重 | —（推导式做不到） | `dict.fromkeys(it)` | pandas `drop_duplicates` | 标准库一行解决 |
| 扁平化 | `[x for sub in it for x in sub]` | `itertools.chain.from_iterable(it)` | NumPy `ravel` | 惰性需求用 chain |
| 词频统计 | `{w: ...count(w) ...}` | `collections.Counter` | — | 生产环境一律用 Counter |
| 分组 | 嵌套推导式（勉强） | `itertools.groupby`（需先排序） | pandas `groupby` | 复杂分组交给专门工具 |
| 惰性管道 | 生成器表达式 | `map` / `filter` / `itertools` 组合 | — | 大数据流式处理首选 |


## 12. 综合实战：RAG 文档入库前的预处理管道

**本章解决什么问题**：把前面所有知识点组合成一个贴近 LLMOps 业务的完整例子——一批原始文档入库前，需要**清洗 → 过滤 → 提取元数据 → 建索引 → 统计关键词**。

### 设计取舍（为什么这么选）

1. **清洗用列表推导式**：数据量小（内存里的一批文档），要的就是立即拿到结果列表；
2. **过滤条件放 `for` 后**：丢弃空文档，元素个数减少；
3. **索引用字典推导式**：`doc_id` 是主键，建一次索引换来后续 O(1) 查找；
4. **词频用 `Counter`**：避免 `count` 的 O(n²) 复杂度（第 10 章模式 4 已验证等价）；
5. **统计总长度用生成器表达式**：纯聚合场景不需要中间列表，省一份内存。


In [30]:
# 综合实战：RAG 文档预处理管道（组合：列表/字典/集合/生成器四种推导式 + Counter）

from collections import Counter

# 原始输入：模拟从数据源拉取的一批文档（含空白标题、空内容等脏数据）
raw_docs = [
    {"doc_id": "d1", "title": "  RAG 检索增强入门 ", "content": "检索 生成 检索 增强"},
    {"doc_id": "d2", "title": "", "content": "这条文档标题为空，应丢弃"},
    {"doc_id": "d3", "title": "Agent 设计模式", "content": "智能体 工具 智能体 规划"},
    {"doc_id": "d4", "title": "   ", "content": "纯空白标题，应丢弃"},
    {"doc_id": "d5", "title": "Memory 组件详解", "content": "记忆 缓冲 记忆 摘要"},
]

# 步骤1【列表推导式 + 过滤】：清洗标题并丢弃空标题文档
clean_docs = [
    {**d, "title": d["title"].strip()}          # 用 ** 解包复制原 dict，覆盖清洗后的 title
    for d in raw_docs
    if d["title"].strip()                        # 过滤：空标题 / 纯空白标题
]
print("步骤1 清洗后保留:", [d["doc_id"] for d in clean_docs])  # ['d1', 'd3', 'd5']

# 步骤2【字典推导式】：构建 doc_id -> 文档 的索引，支撑后续 O(1) 查找
doc_index = {d["doc_id"]: d for d in clean_docs}
print("步骤2 索引键:", list(doc_index.keys()))

# 步骤3【列表推导式 + 条件表达式】：按内容长度打标（映射，元素个数不变）
length_labels = ["长文" if len(d["content"]) > 12 else "短文" for d in clean_docs]
print("步骤3 长度标签:", length_labels)

# 步骤4【集合推导式】：收集语料中出现过哪些去重后的关键词种类
all_keywords = {kw for d in clean_docs for kw in d["content"].split()}
print("步骤4 关键词种类:", sorted(all_keywords))

# 步骤5【生成器表达式 + sum】：惰性统计总字数，不落地中间列表
total_chars = sum(len(d["content"]) for d in clean_docs)
print("步骤5 总字数:", total_chars)

# 步骤6【Counter】：全局关键词词频（替代 count 的 O(n^2) 写法）
freq = Counter(kw for d in clean_docs for kw in d["content"].split())
print("步骤6 高频词 Top2:", freq.most_common(2))


步骤1 清洗后保留: ['d1', 'd3', 'd5']
步骤2 索引键: ['d1', 'd3', 'd5']
步骤3 长度标签: ['短文', '长文', '短文']
步骤4 关键词种类: ['增强', '工具', '摘要', '智能体', '检索', '生成', '缓冲', '规划', '记忆']
步骤5 总字数: 35
步骤6 高频词 Top2: [('检索', 2), ('智能体', 2)]


## 13. 常见陷阱汇总

**本章解决什么问题**：把全文陷阱浓缩成一张速查表，下面的代码单元逐个复现。

| # | 陷阱 | 症状 | 解法 |
| --- | --- | --- | --- |
| 1 | 普通 `for` 变量泄漏 | 循环后外部同名变量被覆盖 | 循环变量用独立命名；或改用推导式 |
| 2 | 生成器只能消费一次 | 第二次遍历静默得到空结果 | 需重复消费就用列表 |
| 3 | 闭包延迟绑定 | 所有 lambda 返回同一个最终值 | 默认参数 `lambda x=x: x` 固定值 |
| 4 | 字典反转键覆盖 | 反转后条目变少、数据丢失 | 反转前确认值唯一 |
| 5 | 过度嵌套 | 推导式超过两层，没人看得懂 | 拆成普通循环 |
| 6 | 推导式里做副作用 | 产出 `[None, ...]` 的无意义列表 | 副作用用普通 `for` |
| 7 | 大列表撑爆内存 | 千万级元素全部驻留内存 | 改生成器表达式 |


In [31]:
# 陷阱复现代码：上表 7 个陷阱逐个验证（输出即证据）

print("=== 陷阱1：for 变量泄漏 ===")
x = 10
for x in range(3):
    pass
print("循环后 x =", x, "（外部变量被覆盖为 2）")

print("\n=== 陷阱2：生成器耗尽 ===")
g = (i for i in range(3))
_ = list(g)
print("第二次遍历:", list(g), "（静默为空，不报错更危险）")

print("\n=== 陷阱3：闭包延迟绑定 ===")
funcs = [lambda: i for i in range(3)]
print("全部返回:", [f() for f in funcs], "（期望 [0,1,2]，实际 [2,2,2]）")

print("\n=== 陷阱4：反转键覆盖 ===")
d = {"a": 1, "b": 1, "c": 2}
inv = {v: k for k, v in d.items()}
print("反转结果:", inv, "（3 条变 2 条，'a' 丢失）")

print("\n=== 陷阱5：过度嵌套（能运行但不可读） ===")
bad = [[a + b for a in range(3) for b in range(3)] for _ in range(2)]
print("嵌套结果长度:", len(bad), "x", len(bad[0]), "—— 超过两层就该拆循环")

print("\n=== 陷阱6：副作用返回值 ===")
ret = [print("x", end=" ") for i in range(2)]
print("-> 收集到:", ret, "（print 返回 None）")

print("\n=== 陷阱7：大列表内存（对比 getsizeof） ===")
import sys
big_list = [i for i in range(1_000_000)]
big_gen = (i for i in range(1_000_000))
print(f"列表 {sys.getsizeof(big_list):,} 字节 vs 生成器 {sys.getsizeof(big_gen):,} 字节")


=== 陷阱1：for 变量泄漏 ===
循环后 x = 2 （外部变量被覆盖为 2）

=== 陷阱2：生成器耗尽 ===
第二次遍历: [] （静默为空，不报错更危险）

=== 陷阱3：闭包延迟绑定 ===
全部返回: [2, 2, 2] （期望 [0,1,2]，实际 [2,2,2]）

=== 陷阱4：反转键覆盖 ===
反转结果: {1: 'b', 2: 'c'} （3 条变 2 条，'a' 丢失）

=== 陷阱5：过度嵌套（能运行但不可读） ===
嵌套结果长度: 2 x 9 —— 超过两层就该拆循环

=== 陷阱6：副作用返回值 ===
x x -> 收集到: [None, None] （print 返回 None）

=== 陷阱7：大列表内存（对比 getsizeof） ===
列表 8,448,728 字节 vs 生成器 192 字节


## 14. 总结

### 通用代码模板（可直接复制）

```python
# 列表推导式：[表达式 for 变量 in 可迭代对象 if 过滤条件]
result = [transform(x) for x in items if keep(x)]

# 字典推导式：{键表达式: 值表达式 for 变量 in 可迭代对象 if 条件}
index = {item["id"]: item for item in items}

# 集合推导式：{表达式 for 变量 in 可迭代对象} —— 自动去重
kinds = {extract(x) for x in items}

# 生成器表达式：(表达式 for 变量 in 可迭代对象) —— 惰性、省内存
total = sum(metric(x) for x in items)
```

### 记忆口诀

1. **方括号列表、花括号字典集合、圆括号生成器**——有冒号是字典，没冒号是集合。
2. **`if` 在 `for` 后是过滤、在前是映射**——前面必须带 `else`。
3. **生成器只走一遍**——要重复用，就建列表。
4. **推导式有自己的作用域**——但类体里读不到类属性。
5. **超过两层就拆循环**——可读性永远优先。
6. **副作用别进推导式**——生成值用推导式，做事情用循环。

### 自测练习题（参考实现见下方代码单元）

1. 给定 `[1, -3, 5, -7, 9]`，用一行推导式得到所有负数的绝对值列表 `[3, 7]`；
2. 给定 `{"a": 1, "b": 2, "c": 3}`，用字典推导式把值全部乘以 10；
3. 给定嵌套列表 `[[1, 2], [3], [4, 5, 6]]`，用推导式扁平化并只保留偶数；
4. 说明为什么 `sum(len(w) for w in words)` 优于 `sum([len(w) for w in words])`；
5. 写出修复闭包延迟绑定的两种思路（提示：默认参数 / 立即调用）。


In [32]:
# 自测练习题 —— 参考实现

# 练习1：过滤 + 映射组合 —— if 在 for 后做过滤，abs 做映射
nums = [1, -3, 5, -7, 9]
answer1 = [abs(x) for x in nums if x < 0]
print("练习1:", answer1)  # [3, 7]

# 练习2：字典值整体变换 —— items() 解包，只改值表达式
d = {"a": 1, "b": 2, "c": 3}
answer2 = {k: v * 10 for k, v in d.items()}
print("练习2:", answer2)  # {'a': 10, 'b': 20, 'c': 30}

# 练习3：扁平化 + 过滤 —— 两个 for 从左到右对应嵌套层次
nested = [[1, 2], [3], [4, 5, 6]]
answer3 = [x for sub in nested for x in sub if x % 2 == 0]
print("练习3:", answer3)  # [2, 4, 6]

# 练习4：生成器不落地中间列表 —— 用 getsizeof 量化说明
import sys
words = ["hello", "world", "python"] * 1000
as_list = [len(w) for w in words]          # 先建 3000 个元素的列表再求和
as_gen = (len(w) for w in words)           # 只有迭代器本身
print("练习4: 中间列表", sys.getsizeof(as_list), "字节 vs 生成器", sys.getsizeof(as_gen), "字节")

# 练习5：闭包延迟绑定的两种修复
fix1 = [lambda x=x: x for x in range(3)]        # 思路1：默认参数在定义时固化值
fix2 = [(lambda v: (lambda: v))(x) for x in range(3)]  # 思路2：外层 lambda 立即调用固值
print("练习5:", [f() for f in fix1], [f() for f in fix2])  # [0, 1, 2] [0, 1, 2]


练习1: [3, 7]
练习2: {'a': 10, 'b': 20, 'c': 30}
练习3: [2, 4, 6]
练习4: 中间列表 26040 字节 vs 生成器 208 字节
练习5: [0, 1, 2] [0, 1, 2]
